# Exploring soil modeling in soilice

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as pl

In [2]:
from soilice.src_soil import MakeDictFloat
from soilice.src_soil import MakeDictArray
from soilice.src_soil import GCEfun
from soilice.src_soil import thetaFun
from soilice.balanceChecks import soilBalanceCheck
from soilice.src_soil import run

ImportError: cannot import name 'GCEfun' from 'soilice.src_soil' (/Users/ani378/USASK/research/projects/FrozenSoilPhysics/soilice/soilice/src_soil.py)

In [ ]:
from animate import animate
from IPython.display import HTML

In [ ]:
# Soil model options:
opts={}
opts['massflag']=1.              # Solve correct equations (set to zero to remove Cdtheta/dt)
opts['gravity']=1.               # 0. for horizontal; 1. for vertical
opts['infiltration']=1.          # 0. for specified psiT; 1. for specified potential infiltration 
opts['cryoflow']=0.              # 0. flow based on psie; 1. flow based on psif 
opts['withadv']=1.               # 0. turn off advection; 1. turn on advection
opts['conductionTop']=1.         # 0. no conduction on upper BC; 1. conduction based on TTop
opts['conductionBot']=1.         # 0. no conduction on lower BC; 1. conduction based on TBot
opts['simulateFlow']=0.          # 0. no flow, constant total water content; 1. simulates flow
opts['simulateTransport']=1.     # 0. no flow, constant total water content; 1. simulates heat transport
opts['freeDrainage']=0.          # 0. no flow lower BC, 1.0 free draining lowerBC
opts['groundHeatFlux']=0.        # 0. no specified ground heat flux; 1. use specified ground heat flux.

In [ ]:
# Import soil parameters:
from soilice.pars_loam import pars

# Import constants:
from soilice.constants import const

# Convert parameter units into days:
pars['Ks']=pars['Ks']*86400
pars['kappa_org']=pars['kappa_org']*86400
pars['kappa_soil']=pars['kappa_soil']*86400
const['kappa_air']=const['kappa_air']*86400
const['kappa_ice']=const['kappa_ice']*86400
const['kappa_liq']=const['kappa_liq']*86400

In [ ]:
# Space grid
zMax=2.
dz=0.02
z=np.arange(dz/2,zMax,dz)
nz=len(z)
dz=np.zeros(nz)+dz

# Time grid
t=np.arange(0,10,0.1) # days
nt=len(t)
dt=t[1]-t[0]

In [ ]:
# Boundary conditions - mass and heat fluxes:
jTopBC=np.zeros(nt) # Specified ground heat flux, if used.

# NOT USED:
qI=np.zeros(nt)
TInf=np.zeros(nt)
TTop=np.zeros(nt)-5.
TBot=np.zeros(nt)+1.

In [ ]:
# Overide the KFun with a custom function:
from numba import jit 
import soilice.src_soil as sm

@jit(nopython=True)
def thermalKfun(psie,psif,T,pars,const):
    thetaL=thetaFun(psif,pars)
    thetaT=thetaFun(psie,pars)
    thetaI=const['rho_liq']/const['rho_ice']*(thetaT-thetaL)
    thetaG=pars['thetaS']-thetaT
    n = pars['thetaS']
    kunfroz=const['kappa_liq']**thetaT*pars['kappa_soil']**(1-n)*const['kappa_air']**thetaG
    kfroz=const['kappa_ice']**thetaT*pars['kappa_soil']**(1-n)*const['kappa_air']**thetaG
    kappa=thetaL/thetaT*kunfroz+thetaI/thetaT*kfroz
    kappa=kappa*1000
    return kappa

sm.thermalKfun=thermalKfun
   # thermalKfun(psie,psif,T,pars,const)

# # # K=KFun(psif,pars)

# # # K=(KFun(psie,pars)+KFun(psif,pars))/2.

In [ ]:
# # For a uniform simulation, just run this code unmodified. This is more efficient:
parsD=MakeDictFloat()
for k in pars: parsD[k]=pars[k]

In [ ]:
# Run scenarios with different initial conditions
psi0=np.linspace(-1,-1,nz) # z-zMax 
T0=np.linspace(1,1,nz)

psie,T1,thetaL1,thetaI1,qT,qB,jT1,jB=sm.run(dt,t,dz,nz,T0,psi0,qI,TTop,TBot,TInf,jTopBC,parsD,const,opts,rtol=1e-8)

In [ ]:
# Run scenarios with different initial conditions
psi0=np.linspace(-1,-1,nz) # z-zMax 
T0=np.linspace(1,1,nz)

psie,T2,thetaL2,thetaI2,qT,qB,jT1,jB=sm.run(dt,t,dz,nz,T0,psi0,qI,TTop,TBot,TInf,jTopBC,parsD,const,opts,rtol=1e-8)

In [ ]:
pl.plot(T1[np.arange(0,len(t),4),:].T,z,'k')
pl.plot(T2[np.arange(0,len(t),4),:].T,z,'r')
pl.ylim(2,0)
pl.show()